# 02 · Train, measure, investigate

**Goal:** implement a small model and training step, then make a claim that is supported by an experiment.

The data generator and experiment harness are provided. You implement the classifier, training step,
and evaluation. The tasks are original to this course; the PyTorch tutorial is your implementation reference.
Everything here runs on the CPU. No dataset downloads or accounts are needed.

In [ ]:
from pathlib import Path
import sys
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "curriculum.json").is_file())
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from workbench import lesson_panel, check
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(2)

display(lesson_panel("02"))

## Inspect the task before choosing a model

Each example has two real-valued features and one of two classes. The label says whether the features
have the same sign. The generator uses separate seeds for training, validation, and test samples.

Use training data to update parameters and validation data to compare choices. Leave the final test
sample untouched until you have selected the model and settings you want to evaluate.

In [ ]:
def make_data(count, seed):
    generator = torch.Generator().manual_seed(seed)
    x = torch.randn(count, 2, generator=generator)
    y = (x[:, 0] * x[:, 1] > 0).long()
    return x, y

train_x, train_y = make_data(400, 101)
valid_x, valid_y = make_data(200, 202)
test_x, test_y = make_data(200, 303)
print("Training features:", tuple(train_x.shape), "labels:", tuple(train_y.shape))
print("Validation features:", tuple(valid_x.shape), "labels:", tuple(valid_y.shape))
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(train_x[:, 0], train_x[:, 1], c=train_y, cmap="coolwarm", s=16, alpha=0.65)
ax.set(xlabel="Feature 1", ylabel="Feature 2", title="Training examples")
plt.show()

**Before training:** could a single linear decision boundary solve this task? What would you expect
a small nonlinear model to do? What accuracy would a constant-class predictor get on these samples?

*Write your prediction here.*

## 1. Build the classifier

Implement `build_classifier()`, returning an `nn.Module` with a small hidden layer and a nonlinearity.
Start with 16 hidden units. It should accept `(batch, 2)` inputs and return `(batch, 2)` raw class scores.
Use the same model structure for your first comparison.

Read **Build Model** in PyTorch's Learn the Basics when you need the module API.

<details><summary>Optional hint</summary>
What do the input and output dimensions of each layer mean? What would happen if all intermediate
transformations were linear? The loss function expects class scores.
</details>

In [ ]:
def build_classifier():
    # Your implementation here.
    return None

In [ ]:
model_ok = check("model", build_classifier)

## 2. Implement one training step

Implement `train_step(model, optimizer, x, y)`. Put the model in training mode and update its parameters
using this batch's cross-entropy loss. Previous batches' gradients should not accumulate.
Return the loss before this update as a Python number.

You can use PyTorch's built-in cross entropy here. Read **Autograd** and **Optimization** for this task.

<details><summary>Optional hint</summary>
Distinguish computing a loss, computing its derivatives, and changing the parameters. Which state
persists from one call to the next? What does your optimizer use when it updates a parameter?
</details>

In [ ]:
def train_step(model, optimizer, x, y):
    # Your implementation here.
    return None

In [ ]:
step_ok = check("step", train_step)

## 3. Evaluate without training

Implement `evaluate(model, x, y)`. Use evaluation mode and disable gradient recording for the forward pass.
Return a dictionary with `loss` (mean cross entropy) and `accuracy` (fraction correct), both Python numbers.
Do not update model parameters. The training step will restore training mode when training resumes.

<details><summary>Optional hint</summary>
How do you turn each vector of class scores into a predicted class? Are evaluation mode and disabling
gradient recording the same operation, or do they serve different purposes?
</details>

In [ ]:
def evaluate(model, x, y):
    # Your implementation here.
    return None

In [ ]:
evaluate_ok = check("evaluate", evaluate)

## 4. Run a baseline

This harness calls your three functions. It deliberately waits until their checks pass.
Training uses the full training set as one batch; each recorded validation point uses the fixed validation set.
The seed controls model initialization. Keep data and seed fixed for the first comparison.

In [ ]:
def run_experiment(seed=7, learning_rate=0.03, epochs=150):
    torch.manual_seed(seed)
    model = build_classifier()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = []
    for epoch in range(epochs):
        batch_loss = train_step(model, optimizer, train_x, train_y)
        if epoch % 10 == 0 or epoch == epochs - 1:
            train_result = evaluate(model, train_x, train_y)
            valid_result = evaluate(model, valid_x, valid_y)
            history.append({"epoch": epoch + 1, "train_loss": train_result["loss"],
                            "valid_loss": valid_result["loss"], "valid_accuracy": valid_result["accuracy"]})
    return {"model": model, "history": history, "seed": seed,
            "learning_rate": learning_rate, "epochs": epochs}

ready = model_ok and step_ok and evaluate_ok
baseline = run_experiment() if ready else None
if baseline:
    print("Baseline settings:", {k: baseline[k] for k in ("seed", "learning_rate", "epochs")})
    print("Final training / validation measurements:", baseline["history"][-1])
else:
    print("Paused: finish the three functions and rerun their checks before training.")

In [ ]:
if baseline:
    rows = baseline["history"]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot([r["epoch"] for r in rows], [r["train_loss"] for r in rows], label="Training")
    ax.plot([r["epoch"] for r in rows], [r["valid_loss"] for r in rows], label="Validation")
    ax.set(xlabel="Epoch", ylabel="Mean cross-entropy loss", title="Baseline learning curves")
    ax.legend()
    plt.show()

## 5. Change one thing

Predict how changing the learning rate will affect the curves. Set `new_learning_rate` below to your choice.
The harness resets the initialization and preserves the data, architecture, and training budget.
This is one controlled comparison; you can check whether it persists across several seeds afterward.

**My prediction and reason:**

*Write before running.*

In [ ]:
new_learning_rate = None  # Choose a positive number after recording your prediction.
variation = None
if ready and new_learning_rate is not None:
    variation = run_experiment(learning_rate=new_learning_rate)
    print("Baseline:", baseline["history"][-1])
    print("Variation:", variation["history"][-1])
else:
    print("Choose a learning rate when you are ready to test your prediction.")

## 6. Evaluate your selected model once

Use the validation comparison to select a model. Set `selected` to `baseline` or `variation` and set
`ready_for_final_test` to `True` when you are ready. If you tune choices after looking at this result,
the sample no longer serves as an untouched final test.

In [ ]:
selected = baseline
ready_for_final_test = False
if ready_for_final_test and selected is not None:
    final_test_result = evaluate(selected["model"], test_x, test_y)
    print("Final held-out test:", final_test_result)
else:
    print("Final test remains unused.")

## What does your experiment establish?

**Claim:** *one sentence.*

**Evidence:** *settings, measurements, and comparison.*

**Alternative explanation or limitation:** *what have you not ruled out?*

**Next experiment:** *what would resolve one uncertainty?*

You are ready for the next module when you can explain the model, training step, evaluation, and one fair comparison.
Tick your completed tasks, leave a stopping-point note, and save the notebook.

Continue to [03 · Inside a transformer](03_Transformer_Internals.ipynb) or return [home](../00_Start_Here.ipynb).